# Lab 14. Filtering and Wavelets for Time Series

[Open this lab in Google Colab](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-14-filtering-wavelets-lab.ipynb)

This lab is designed for **independent study**. It gives the necessary mathematical background before each programming task, then asks you to interpret the output. You do not need prior experience with signal processing beyond basic linear algebra, Fourier ideas, and the earlier time-series labs.

## Learning goals

By the end of this lab, you should be able to:

1. Explain a **linear filter** as a weighted sum of shifted observations.
2. Distinguish **causal filters** from centered filters.
3. Use convolution to implement smoothing, differencing, and high-pass filtering.
4. Interpret the **frequency response** of a filter.
5. Compare time-domain filtering with periodogram-based frequency-domain interpretation.
6. Understand why Fourier methods are global and why wavelets help with local, transient behavior.
7. Implement a simple **Haar wavelet transform** from scratch.
8. Use wavelet coefficients for denoising and transient detection.

## Why this lab matters

A time series may contain slow trend, oscillations, noise, and sudden local events. Classical Fourier analysis is powerful for identifying dominant global frequencies. But if a signal changes behavior only for a short time, global Fourier coefficients can spread that information over many frequencies. Filtering and wavelets provide tools for separating slow and fast components and for detecting time-localized structure.

## 0. Mathematical background before programming

### 0.1 Linear filters

A linear filter transforms an input time series $x_t$ into an output series $y_t$ by a weighted sum of shifted values:

$$
y_t = \sum_j a_j x_{t-j}.
$$

The coefficients $a_j$ are the **filter weights**. When the same weights are used at every time $t$, the filter is **time invariant**.

A filter is **causal** if it uses only the present and past:

$$
a_j = 0 \quad \text{for } j < 0.
$$

For example, the causal moving average

$$
y_t = \frac{1}{3}(x_t + x_{t-1} + x_{t-2})
$$

uses only current and past observations. A centered moving average

$$
y_t = \frac{1}{3}(x_{t-1} + x_t + x_{t+1})
$$

is useful for smoothing historical data, but it is not causal because it uses a future value $x_{t+1}$.

### 0.2 Impulse response

If the input is a unit impulse,

$$
x_t = \begin{cases}1, & t=0,\\ 0, & t\ne 0,\end{cases}
$$

then the output reveals the filter weights. This is why the sequence of filter coefficients is also called the **impulse response**.

### 0.3 Frequency response

A linear time-invariant filter changes the strength of each frequency. If the input has spectral density $f_x(\omega)$ and the filter has frequency response $A(\omega)$, then the output spectrum is approximately

$$
f_y(\omega) = |A(\omega)|^2 f_x(\omega).
$$

Thus:

- a **low-pass filter** keeps low frequencies and suppresses high frequencies;
- a **high-pass filter** suppresses slow components and emphasizes rapid changes;
- a **band-pass filter** emphasizes a middle range of frequencies.

### 0.4 Wavelet idea

Fourier methods use sine and cosine waves extending over the full time interval. Wavelets use short, localized building blocks. They are useful when a time series has local jumps, bursts, or time-varying frequency.

In this lab we implement the simplest wavelet transform: the **Haar transform**. It repeatedly computes local averages and local differences:

$$
avg_k = \frac{x_{2k} + x_{2k+1}}{\sqrt{2}}, \qquad
coef_k = \frac{x_{2k} - x_{2k+1}}{\sqrt{2}}.
$$

The averages describe coarse structure. The detail coefficients describe local changes.

## 1. Setup

Run the next cell first. We use only standard scientific Python packages available in Google Colab. We implement the Haar wavelet transform ourselves, so no extra wavelet package is required.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import signal

rng = np.random.default_rng(7339)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

print("Libraries loaded.")

## 2. Helper functions

These functions will be reused throughout the lab.

### Important implementation choices

- `periodogram` computes a simple one-sided periodogram using the FFT.
- `centered_moving_average` uses a symmetric window and is not causal.
- `causal_moving_average` uses only current and past values.
- `frequency_response` evaluates a filter on Fourier frequencies.

In [ ]:
def periodogram(x, dt=1.0):
    """Return positive Fourier frequencies and a simple one-sided periodogram."""
    x = np.asarray(x, dtype=float)
    x_centered = x - x.mean()
    n = len(x_centered)
    fft_vals = np.fft.rfft(x_centered)
    freqs = np.fft.rfftfreq(n, d=dt)
    power = (np.abs(fft_vals) ** 2) / n
    return freqs, power


def plot_series(x, title, xlabel="time", ylabel="value"):
    plt.figure(figsize=(10, 4))
    plt.plot(np.arange(len(x)), x)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.show()


def plot_periodogram(x, title, max_freq=None):
    freqs, power = periodogram(x)
    if max_freq is not None:
        mask = freqs <= max_freq
        freqs = freqs[mask]
        power = power[mask]
    plt.figure(figsize=(10, 4))
    plt.plot(freqs, power)
    plt.title(title)
    plt.xlabel("frequency: cycles per time unit")
    plt.ylabel("periodogram power")
    plt.show()


def centered_moving_average(x, window):
    """Centered moving average with edge padding."""
    if window % 2 == 0:
        raise ValueError("Use an odd window size for a centered moving average.")
    weights = np.ones(window) / window
    pad = window // 2
    x_pad = np.pad(x, (pad, pad), mode="edge")
    return np.convolve(x_pad, weights, mode="valid")


def causal_moving_average(x, window):
    """Causal moving average using only current and past values."""
    y = np.zeros_like(x, dtype=float)
    for t in range(len(x)):
        start = max(0, t - window + 1)
        values = x[start:t + 1]
        y[t] = values.mean()
    return y


def apply_filter_same_length(x, weights):
    """Apply a centered convolution and return an output of the same length."""
    return np.convolve(x, weights, mode="same")


def frequency_response(weights, n_freq=512):
    """Compute the frequency response magnitude squared for real filter weights."""
    weights = np.asarray(weights, dtype=float)
    freqs = np.linspace(0, 0.5, n_freq)
    j = np.arange(len(weights))
    response = np.array([np.sum(weights * np.exp(-2j * np.pi * f * j)) for f in freqs])
    return freqs, response, np.abs(response) ** 2

## 3. Simulate a signal with slow, fast, local, and noisy components

We create a synthetic signal with four parts:

1. a slow component,
2. a seasonal component,
3. a short burst that occurs only locally,
4. random noise.

This is useful because we know the truth. A good filtering method should separate these components in a way that matches the design of the data.

In [ ]:
n = 512
t = np.arange(n)

slow = 1.2 * np.sin(2 * np.pi * t / 160)
seasonal = 0.7 * np.sin(2 * np.pi * t / 24)
local_burst = np.zeros(n)
burst_region = (t >= 250) & (t < 310)
local_burst[burst_region] = 1.2 * np.sin(2 * np.pi * t[burst_region] / 6)
noise = rng.normal(0, 0.45, size=n)

x = slow + seasonal + local_burst + noise

components = pd.DataFrame({
    "slow": slow,
    "seasonal": seasonal,
    "local_burst": local_burst,
    "noise": noise,
    "observed": x
})

components.head()

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(t, x, label="observed", linewidth=1.2)
plt.plot(t, slow, label="slow component", linewidth=2)
plt.plot(t, seasonal, label="seasonal component", alpha=0.8)
plt.plot(t, local_burst, label="local burst", alpha=0.8)
plt.title("Synthetic signal with slow trend, seasonality, local burst, and noise")
plt.xlabel("time")
plt.ylabel("value")
plt.legend()
plt.show()

plot_periodogram(x, "Periodogram of the observed signal", max_freq=0.25)

### Checkpoint 1

Look at the time plot and the periodogram.

1. Which component is easy to see in the time plot?
2. Which component is easier to see in the periodogram?
3. Why might the local burst be hard to summarize using only a global periodogram?

## 4. Moving-average filters as low-pass filters

A moving average reduces rapid fluctuations by replacing each value with a local average. This is a low-pass filter because it keeps slowly changing components and suppresses rapidly changing components.

We compare a centered moving average and a causal moving average.

In [ ]:
window = 25
x_centered = centered_moving_average(x, window=window)
x_causal = causal_moving_average(x, window=window)

plt.figure(figsize=(11, 5))
plt.plot(t, x, label="observed", alpha=0.45)
plt.plot(t, x_centered, label=f"centered moving average, window={window}", linewidth=2)
plt.plot(t, x_causal, label=f"causal moving average, window={window}", linewidth=2)
plt.title("Centered versus causal smoothing")
plt.xlabel("time")
plt.ylabel("value")
plt.legend()
plt.show()

### Interpretation

The centered moving average has less lag because it uses future and past values. The causal moving average is appropriate for online forecasting because it only uses the current and past observations, but it tends to lag behind turning points.

In [ ]:
plot_periodogram(x, "Raw observed series: periodogram", max_freq=0.25)
plot_periodogram(x_centered, "Centered moving average: periodogram", max_freq=0.25)
plot_periodogram(x - x_centered, "High-pass residual: raw minus centered moving average", max_freq=0.25)

### Checkpoint 2

The high-pass residual is

$$
r_t = x_t - \widehat{m}_t.
$$

Answer these questions:

1. Which frequencies remain strong after smoothing?
2. Which frequencies remain strong in the high-pass residual?
3. Why does smoothing reduce noise but also blur sharp local features?

## 5. Frequency response of common filters

A filter can be understood by its effect on different frequencies.

We compare four filters:

1. 3-point moving average,
2. 25-point moving average,
3. first difference $y_t = x_t - x_{t-1}$,
4. second difference $y_t = x_t - 2x_{t-1} + x_{t-2}$.

Moving averages are low-pass filters. Difference filters are high-pass filters because they suppress slow changes and emphasize rapid changes.

In [ ]:
filters = {
    "MA(3)": np.ones(3) / 3,
    "MA(25)": np.ones(25) / 25,
    "first difference": np.array([1, -1]),
    "second difference": np.array([1, -2, 1])
}

plt.figure(figsize=(10, 5))
for name, weights in filters.items():
    freqs, response, power = frequency_response(weights)
    plt.plot(freqs, power, label=name)
plt.title("Power transfer functions of common filters")
plt.xlabel("frequency: cycles per time unit")
plt.ylabel("squared gain")
plt.legend()
plt.show()

### Checkpoint 3

Use the figure above to answer:

1. Which filter most strongly suppresses high frequencies?
2. Which filters suppress frequency zero?
3. Why does the first-difference filter remove a constant trend level?

## 6. Filtering a signal using difference filters

The first difference is a simple high-pass filter:

$$
y_t = x_t - x_{t-1}.
$$

It removes constant level and makes slow motion less dominant. The second difference emphasizes curvature and local changes even more.

In [ ]:
first_diff = np.diff(x, prepend=x[0])
second_diff = np.diff(first_diff, prepend=first_diff[0])

plt.figure(figsize=(11, 5))
plt.plot(t, x, label="observed", alpha=0.4)
plt.plot(t, first_diff, label="first difference", alpha=0.8)
plt.plot(t, second_diff, label="second difference", alpha=0.8)
plt.axvspan(250, 310, alpha=0.15, label="true local burst region")
plt.title("High-pass filtering by differencing")
plt.xlabel("time")
plt.ylabel("value")
plt.legend()
plt.show()

plot_periodogram(first_diff, "Periodogram of first difference", max_freq=0.25)
plot_periodogram(second_diff, "Periodogram of second difference", max_freq=0.25)

### Checkpoint 4

Differencing can help expose local bursts, but it can also amplify noise.

1. Which version makes the burst most visible?
2. Which version looks noisiest?
3. In forecasting, why might over-differencing be dangerous?

## 7. Short-time Fourier transform: a bridge to wavelets

A global periodogram tells us which frequencies are present, but not **when** they occur. A short-time Fourier transform (STFT) computes Fourier power in moving windows.

This is not a wavelet transform, but it motivates time-frequency analysis.

In [ ]:
f_stft, t_stft, Zxx = signal.stft(x, fs=1.0, window="hann", nperseg=64, noverlap=48)
power_stft = np.abs(Zxx) ** 2

plt.figure(figsize=(10, 5))
plt.pcolormesh(t_stft, f_stft, power_stft, shading="auto")
plt.colorbar(label="local spectral power")
plt.ylim(0, 0.25)
plt.axvspan(250, 310, color="white", alpha=0.15, label="burst region")
plt.title("Short-time Fourier transform: local frequency content")
plt.xlabel("time")
plt.ylabel("frequency")
plt.legend(loc="upper right")
plt.show()

### Checkpoint 5

The local burst has period about 6, so its frequency is about $1/6 \approx 0.167$.

1. Do you see extra power near frequency 0.167 around the burst time?
2. Why does the STFT depend on window length?
3. What tradeoff occurs between time resolution and frequency resolution?

## 8. Haar wavelet transform from scratch

The Haar transform repeatedly separates a sequence into local averages and local differences.

For pairs $(x_0,x_1)$, $(x_2,x_3)$, and so on:

$$
avg_k = \frac{x_{2k}+x_{2k+1}}{\sqrt{2}},
\qquad
coef_k = \frac{x_{2k}-x_{2k+1}}{\sqrt{2}}.
$$

Then we apply the same operation to the averages. This creates detail coefficients at multiple scales.

We will implement both the forward and inverse Haar transform.

In [ ]:
def haar_dwt(x):
    """Orthogonal Haar wavelet transform for a vector whose length is a power of 2."""
    x = np.asarray(x, dtype=float).copy()
    n = len(x)
    if n & (n - 1) != 0:
        raise ValueError("Length must be a power of 2.")
    coeffs = []
    current = x.copy()
    while len(current) > 1:
        avg = (current[0::2] + current[1::2]) / np.sqrt(2)
        detail = (current[0::2] - current[1::2]) / np.sqrt(2)
        coeffs.append(detail)
        current = avg
    coeffs.append(current)
    return coeffs


def haar_idwt(coeffs):
    """Inverse Haar transform from coefficient list returned by haar_dwt."""
    current = coeffs[-1]
    for detail in reversed(coeffs[:-1]):
        out = np.empty(detail.size * 2)
        out[0::2] = (current + detail) / np.sqrt(2)
        out[1::2] = (current - detail) / np.sqrt(2)
        current = out
    return current


def flatten_haar_coeffs(coeffs):
    """Return all coefficients in a convenient plotting order."""
    return np.concatenate([coeffs[-1]] + list(reversed(coeffs[:-1])))

# Test exact reconstruction.
example = np.array([2.0, 4.0, 1.0, 3.0, 8.0, 6.0, 5.0, 7.0])
coeffs_example = haar_dwt(example)
reconstructed_example = haar_idwt(coeffs_example)

print("Original:     ", example)
print("Reconstructed:", np.round(reconstructed_example, 6))
print("Max reconstruction error:", np.max(np.abs(example - reconstructed_example)))

### Checkpoint 6

The reconstruction error should be essentially zero. This means the Haar transform did not lose information. It only represented the same signal in a new coordinate system.

## 9. Haar coefficients by scale

We now apply the Haar transform to the observed signal. Since our synthetic signal has length 512, which is a power of 2, it is ready for the transform.

The detail coefficients at small scales capture rapid local changes. Coarser scales capture broader changes.

In [ ]:
coeffs = haar_dwt(x)
x_recon = haar_idwt(coeffs)
print("Max reconstruction error:", np.max(np.abs(x - x_recon)))

energy_by_scale = []
labels = []
for level, detail in enumerate(coeffs[:-1], start=1):
    energy_by_scale.append(np.sum(detail ** 2))
    labels.append(f"detail L{level}")
energy_by_scale.append(np.sum(coeffs[-1] ** 2))
labels.append("final average")

energy_table = pd.DataFrame({
    "component": labels,
    "energy": energy_by_scale,
    "fraction": np.array(energy_by_scale) / np.sum(x ** 2)
})
energy_table

In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(np.arange(len(energy_table)), energy_table["fraction"])
plt.xticks(np.arange(len(energy_table)), energy_table["component"], rotation=45, ha="right")
plt.title("Fraction of signal energy by Haar scale")
plt.ylabel("fraction of total energy")
plt.tight_layout()
plt.show()

flat_coeffs = flatten_haar_coeffs(coeffs)
plt.figure(figsize=(10, 4))
plt.plot(flat_coeffs, linewidth=1)
plt.title("Haar transform coefficients")
plt.xlabel("coefficient index")
plt.ylabel("coefficient value")
plt.show()

### Checkpoint 7

1. Which scale has the most energy?
2. Do the fine-scale detail coefficients look sparse or dense?
3. Why might sparse detail coefficients be useful for denoising?

## 10. Wavelet denoising by thresholding

Noise often creates many small detail coefficients. A simple denoising strategy is:

1. compute wavelet coefficients,
2. shrink small detail coefficients toward zero,
3. reconstruct the signal.

We use soft thresholding:

$$
S_\lambda(c) = \operatorname{sign}(c) \max(|c|-\lambda, 0).
$$

This keeps large coefficients and shrinks small coefficients.

In [ ]:
def soft_threshold(c, lam):
    return np.sign(c) * np.maximum(np.abs(c) - lam, 0.0)


def haar_denoise(x, lam):
    coeffs = haar_dwt(x)
    new_coeffs = []
    for detail in coeffs[:-1]:
        new_coeffs.append(soft_threshold(detail, lam))
    new_coeffs.append(coeffs[-1])
    return haar_idwt(new_coeffs), new_coeffs

# Estimate noise scale from the finest detail coefficients.
finest = coeffs[0]
sigma_hat = np.median(np.abs(finest - np.median(finest))) / 0.6745
universal_threshold = sigma_hat * np.sqrt(2 * np.log(n))

x_denoised, coeffs_denoised = haar_denoise(x, lam=universal_threshold)

print("Estimated noise sigma:", round(sigma_hat, 3))
print("Universal threshold:", round(universal_threshold, 3))

plt.figure(figsize=(11, 5))
plt.plot(t, x, label="observed", alpha=0.45)
plt.plot(t, slow + seasonal + local_burst, label="true noiseless signal", linewidth=2)
plt.plot(t, x_denoised, label="Haar denoised", linewidth=2)
plt.axvspan(250, 310, alpha=0.10, label="burst region")
plt.title("Wavelet denoising by soft thresholding")
plt.xlabel("time")
plt.ylabel("value")
plt.legend()
plt.show()

rmse_raw = np.sqrt(np.mean((x - (slow + seasonal + local_burst)) ** 2))
rmse_denoised = np.sqrt(np.mean((x_denoised - (slow + seasonal + local_burst)) ** 2))

print("Raw RMSE against noiseless truth:", round(rmse_raw, 3))
print("Denoised RMSE against noiseless truth:", round(rmse_denoised, 3))

### Checkpoint 8

1. Did thresholding improve RMSE in this simulation?
2. Did the denoised curve preserve the local burst?
3. What can go wrong if the threshold is too large?

## 11. Wavelet details for transient detection

Wavelet detail coefficients can help identify local events. Here we reconstruct the signal using only one detail scale at a time. This lets us see which scale responds to the burst.

In [ ]:
def reconstruct_one_detail(coeffs, detail_index):
    """Reconstruct using only one detail component. detail_index starts at 0 for finest."""
    new_coeffs = []
    for i, detail in enumerate(coeffs[:-1]):
        if i == detail_index:
            new_coeffs.append(detail.copy())
        else:
            new_coeffs.append(np.zeros_like(detail))
    new_coeffs.append(np.zeros_like(coeffs[-1]))
    return haar_idwt(new_coeffs)

max_detail_to_plot = 5
plt.figure(figsize=(11, 6))
for i in range(max_detail_to_plot):
    detail_recon = reconstruct_one_detail(coeffs, i)
    plt.plot(t, detail_recon + 3 * i, label=f"detail level {i+1}")
plt.axvspan(250, 310, alpha=0.12, label="burst region")
plt.title("Reconstructed Haar detail components by scale")
plt.xlabel("time")
plt.ylabel("offset detail component")
plt.legend(loc="upper right")
plt.show()

### Checkpoint 9

1. Which detail level responds most strongly to the local burst?
2. Which detail levels look mostly like noise?
3. Why might multi-scale features be useful for anomaly detection?

## 12. Compare moving-average denoising and wavelet denoising

Moving averages are simple and useful, but they blur local events. Wavelet thresholding can preserve sharp local changes better, depending on the signal and threshold.

In [ ]:
x_ma_denoised = centered_moving_average(x, window=9)
truth = slow + seasonal + local_burst

rmse_ma = np.sqrt(np.mean((x_ma_denoised - truth) ** 2))
rmse_wavelet = np.sqrt(np.mean((x_denoised - truth) ** 2))

comparison = pd.DataFrame({
    "method": ["raw observed", "centered moving average", "Haar wavelet thresholding"],
    "RMSE against noiseless truth": [rmse_raw, rmse_ma, rmse_wavelet]
})
comparison

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(t, truth, label="true noiseless signal", linewidth=2)
plt.plot(t, x, label="observed", alpha=0.3)
plt.plot(t, x_ma_denoised, label="moving average denoised", linewidth=2)
plt.plot(t, x_denoised, label="wavelet denoised", linewidth=2)
plt.xlim(220, 340)
plt.axvspan(250, 310, alpha=0.10, label="burst region")
plt.title("Zoom near the local burst")
plt.xlabel("time")
plt.ylabel("value")
plt.legend()
plt.show()

### Checkpoint 10

Look carefully at the zoomed plot.

1. Which method preserves the burst better?
2. Which method looks smoother outside the burst?
3. Which method would you prefer for trend estimation? Which for transient detection?

## 13. Student practice: choose your own filter

In this practice section, change the filter weights and interpret the result.

Try these examples:

- a 5-point moving average: `[1, 1, 1, 1, 1] / 5`
- a weighted moving average: `[1, 2, 3, 2, 1] / 9`
- a high-pass filter: `[1, -1]`
- a curvature filter: `[1, -2, 1]`
- an edge-like filter: `[-1, 0, 1]`

In [ ]:
# Try changing these weights.
student_weights = np.array([1, 2, 3, 2, 1], dtype=float)
student_weights = student_weights / student_weights.sum()

student_filtered = apply_filter_same_length(x, student_weights)
freqs, response, power_gain = frequency_response(student_weights)

plt.figure(figsize=(11, 4))
plt.plot(t, x, label="observed", alpha=0.4)
plt.plot(t, student_filtered, label="student filter output", linewidth=2)
plt.title("Output of your chosen filter")
plt.xlabel("time")
plt.ylabel("value")
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(freqs, power_gain)
plt.title("Power transfer function of your chosen filter")
plt.xlabel("frequency")
plt.ylabel("squared gain")
plt.show()

### Practice questions

1. Is your filter low-pass, high-pass, or something else?
2. Is your filter causal as written? Explain.
3. Does the filter preserve the local burst or smooth it away?

## 14. AI-assisted study prompts

Use an AI assistant as a tutor, not as a replacement for your own reasoning. Good prompts ask for diagnosis, explanation, and verification.

### Prompt 1: frequency response explanation

> I applied a 25-point moving average to a time series. Explain why this is a low-pass filter. Then tell me how to verify the claim using the periodogram before and after filtering.

### Prompt 2: wavelet denoising critique

> I denoised a time series using Haar wavelet soft thresholding. The signal contains a short burst. What diagnostics should I check to make sure the burst was not removed as noise?

### Prompt 3: leakage check

> I used a centered moving average before forecasting. Explain why this may create data leakage. How should I modify the filter for online forecasting?

### Prompt 4: choose a method

> I have a time series with slow trend, strong seasonality, sudden short-lived shocks, and random noise. Compare moving-average filtering, Fourier analysis, STFT, and wavelet analysis for this task.

## 15. Exercises

### Exercise 1. Causal versus centered smoothing

Create a simulated time series with a sudden jump. Compare a centered moving average and a causal moving average. Which one reacts faster? Which one is valid for real-time forecasting?

### Exercise 2. Window length and frequency response

Plot the power transfer functions for moving-average windows of length 3, 7, 15, and 31. How does increasing the window size change the filter?

### Exercise 3. Difference filters

Apply the first-difference and second-difference filters to a slowly trending signal. Explain why these filters emphasize high frequencies.

### Exercise 4. STFT window length

Repeat the STFT example with `nperseg=32`, `64`, and `128`. Describe the time-frequency tradeoff.

### Exercise 5. Wavelet threshold tuning

Replace the universal threshold by `0.25 * universal_threshold`, `0.5 * universal_threshold`, and `2 * universal_threshold`. Which threshold gives the best visual result? Which gives the best RMSE in this simulation?

### Exercise 6. Build wavelet features

For rolling windows of length 64 from the simulated signal, compute the energy in the first three Haar detail levels. Plot the feature values over time. Do they help locate the burst?

## 16. Mini-project

Choose one of the following mini-projects.

### Option A: Filter design for seasonal data

Simulate a time series with trend, weekly seasonality, and noise. Design a filter to estimate the trend while suppressing weekly seasonality. Explain your filter using both time-domain plots and frequency response plots.

### Option B: Transient detection by wavelets

Simulate several short bursts at unknown times. Use Haar detail coefficients to create an anomaly score. Compare your wavelet anomaly score with a rolling standard deviation anomaly score.

### Option C: Fourier versus wavelet summary

Create a signal whose frequency changes halfway through the sample. Compare the global periodogram, STFT, and Haar wavelet details. Explain which method best reveals the time-varying behavior.

## 17. Lab checklist

Before you finish, make sure you can do the following without looking at the solutions:

- [ ] Define a linear filter.
- [ ] Explain causal versus non-causal filtering.
- [ ] Implement a moving average using convolution.
- [ ] Explain why moving averages are low-pass filters.
- [ ] Explain why differencing is a high-pass filter.
- [ ] Compute and interpret a power transfer function.
- [ ] Use a periodogram to compare raw and filtered signals.
- [ ] Explain why global Fourier analysis can miss localized behavior.
- [ ] Implement a Haar wavelet transform from scratch.
- [ ] Use wavelet coefficients for denoising or transient detection.

You are now ready to connect filtering and wavelets to modern time-series feature extraction, denoising, and anomaly detection.